In [0]:
spark.sql("SELECT 'connected' as status").show()

In [0]:
# Create a database to hold all Bank Pulse tables
spark.sql("CREATE DATABASE IF NOT EXISTS bank_pulse")
spark.sql("USE bank_pulse")
print("Database ready")

In [0]:
import requests
import pandas as pd
import time
from pyspark.sql import functions as F
from datetime import datetime

def fetch_fdic_financials(limit=1000, max_records=50000):
    base_url = "https://banks.data.fdic.gov/api/financials"
    
    fields = [
        "REPDTE", "CERT", "INTINC", "EINTEXP",
        "NIEXP", "NETINC", "ASSET", "DEP",
        "LNLSNET", "NPERFV", "RBC1RWAJ", "ROA", "ROE"
    ]
    
    all_records = []
    offset = 0
    
    while offset < max_records:
        params = {
            "fields": ",".join(fields),
            "limit": limit,
            "offset": offset,
            "sort_by": "REPDTE",
            "sort_order": "DESC",
            "output": "json"
        }
        
        # Retry loop with backoff
        for attempt in range(5):
            try:
                response = requests.get(base_url, params=params, timeout=30)
                response.raise_for_status()
                break
            except requests.exceptions.HTTPError as e:
                if response.status_code == 429:
                    wait = 2 ** attempt  # 1s, 2s, 4s, 8s, 16s
                    print(f"Rate limited. Waiting {wait}s before retry {attempt+1}/5...")
                    time.sleep(wait)
                else:
                    raise
        
        data = response.json()
        records = data.get("data", [])
        if not records:
            break
        
        all_records.extend([r["data"] for r in records])
        offset += limit
        
        print(f"Fetched {len(all_records):,} records...")
        
        time.sleep(0.5)  # 500ms pause between every request
        
        if len(records) < limit:
            break
    
    return all_records

# Fetch and write to Delta
print("Starting ingestion...")
records = fetch_fdic_financials(limit=1000, max_records=50000)
print(f"Total: {len(records):,} records")

df = spark.createDataFrame(pd.DataFrame(records))
df = df.withColumn("ingested_at", F.current_timestamp()) \
       .withColumn("source", F.lit("fdic_bankfind_api")) \
       .withColumn("batch_id", F.lit(datetime.now().strftime("%Y%m%d_%H%M%S")))

df.write.format("delta").mode("overwrite") \
  .partitionBy("REPDTE") \
  .saveAsTable("bank_pulse.bronze_financials")

print("✓ Done")

In [0]:
spark.table("bank_pulse.bronze_financials").printSchema()